<a href="https://colab.research.google.com/github/henry09027/NTU-Course-MachineLearning/blob/main/Homework/HW1/Code/2026ML_HW1_D15949002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **2026 ML FALL HW1: PM2.5 Prediction (Regression)**

Author: MLTAs

Methods:
* Training with all data
* Optimizer: RMSProp (default)

**Part 1 - main pipeline** (all TODOs completed): `valid()`, `train_config`, `minibatch_2()` (2nd-order polynomial), `normalize_train_data()`, feature selection. Running Part 1 produces `my_sol.csv` (features PM10 + PM2.5, z-score, `train_config`).

**Part 2 - experiments for the report** (Q1 / Q2 / Q3): every experiment is listed with its parameters and the Kaggle public RMSE that was obtained, and writes its own csv. The two best submissions are marked at the end.

# **Part 1 - Main pipeline**

## Imports & random seed (DO NOT modify)

In [1]:
import numpy as np
import csv
import math
import pandas as pd
import os

seed = 9487
np.random.seed(seed)

## Download & load data

In [2]:
!gdown --id "1Hfzrcm69QwdFvdeF0uASoQlcVxKw_hHy" --output "train.csv"
!gdown --id '155N6fzI7vAFzHAGdy6jkaWIksWH6Y1G2' --output "test.csv"

# Incase the links above die, you can use the following instead.
#!gdown --id '11abE854Eyv4BA7qt5k8r_80sJ3KuOQUN' --output "train.csv"
#!gdown --id '1uod-Z4ztluXnuHtgUbm39nMudUKqXHMl' --output "test.csv"

# If the data is still missing, you can manually download it from kaggle, and upload the files under /content

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Hfzrcm69QwdFvdeF0uASoQlcVxKw_hHy
To: /content/train.csv
100% 324k/324k [00:00<00:00, 21.4MB/s]
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=155N6fzI7vAFzHAGdy6jkaWIksWH6Y1G2
To: /content/test.csv
100% 49.0k/49.0k [00:00<00:00, 12.2MB/s]


## Data preprocessing: `valid()` and `parse2train()`

In [3]:
def valid(x, y):
  # Filter out extreme values. In test.csv every PM2.5 value (inputs) is <= 30, so training windows containing
  # PM2.5 spikes (train goes up to 80) come from a different regime and only distort the fit.
  # Keep a window only if the target and all 8 input PM2.5 hours are <= 30.
  # (feats contains only PM2.5, which is NOT normalized, so x.max() / y are in raw PM2.5 units here.)
  return y <= 30 and x.max() <= 30


# Create your dataset
def parse2train(data, feats):

  x = []
  y = []

  # Use data #0~#7 to predict #8 => Total data length should be decresased by 8.
  total_length = data.shape[1] - 8

  for i in range(total_length):
    x_tmp = data[feats, i:i+8] # Use data #0~#7 to predict #8, data #1~#8 to predict #9, etc.
    y_tmp = data[-1, i+8] # last column of (i+8)th row: PM2.5

    # Filter out extreme values to train.
    if valid(x_tmp, y_tmp):
      x.append(x_tmp.reshape(-1,))
      y.append(y_tmp)

  # x.shape: (n, 15, 8)
  # y.shape: (n, 1)
  x = np.array(x)
  y = np.array(y)

  return x,y

## Gradient descent: RMSProp `minibatch()` and 2nd-order polynomial `minibatch_2()`
###**RMSProp**
1. $v_t=\beta \cdot v_{t-1} + (1-\beta)(\nabla w_t)^2$
2. $w_{t+1}=w_t - \frac{\eta}{\sqrt{(v_t)}+\epsilon}\nabla w_t$




* This is our gradient descent algorithm. RMSProp was implemented in `minibatch()`.
* You can implement other algorithm, such as SGD or other gradient descent variants listed below, which may (or may not) improve performance.
* However, **modules like sklearn and pytorch are not allowed!!!**
* Ref:
  - Prof. G. Hinton's lecture: https://www.cs.toronto.edu/~tijmen/csc321/slides/lecture_slides_lec6.pdf
  - Prof. Hung-Yi Lee's video: https://youtu.be/HYUXEeh3kwY?si=RtLjSj51WK1pmz87

###**Adam (RMSProp + Momemtum)**
* Ref:
  - Paper: https://arxiv.org/pdf/1412.6980
  - Prof. Hung-Yi Lee's video: https://youtu.be/HYUXEeh3kwY?si=RtLjSj51WK1pmz87

###**AdamW (Adam with decoupled weight decay)**
* Ref:
  - Paper: https://arxiv.org/pdf/1711.05101

In [4]:
def minibatch(x, y, config):
    # Randomize the data in minibatch
    index = np.arange(x.shape[0])
    np.random.shuffle(index)
    x = x[index]
    y = y[index]

    # Initialization
    batch_size = config.batch_size
    lr = config.lr
    epoch = config.epoch
    decay_rate = config.decay_rate
    epsilon = 1e-8

    # Linear regression: only contains two parameters (w, b).
    w = np.full(x[0].shape, 0.1).reshape(-1, 1)
    bias = 0.1

    # Optimizer states
    cache_w = np.zeros_like(w)
    cache_b = 0.0

    # Training loop
    for num in range(epoch):
        for b in range(int(x.shape[0] / batch_size)):
            x_batch = x[b * batch_size:(b + 1) * batch_size]
            y_batch = y[b * batch_size:(b + 1) * batch_size].reshape(-1, 1)

            # Prediction of linear regression
            pred = np.dot(x_batch, w) + bias

            # Loss
            loss = y_batch - pred

            # Compute gradient
            g_t = np.dot(x_batch.transpose(), loss) * (-2)
            g_t_b = loss.sum(axis=0) * (-2)

            # Update cache
            cache_w = decay_rate * cache_w + (1 - decay_rate) * g_t**2
            cache_b = decay_rate * cache_b + (1 - decay_rate) * g_t_b**2

            # Update weight & bias
            w -= lr * g_t / (np.sqrt(cache_w) + epsilon)
            bias -= lr * g_t_b / (np.sqrt(cache_b) + epsilon)

    return w, bias

# 2nd-order polynomial regression (no interaction terms): y = b + sum_i (w1_i * x_i + w2_i * x_i^2)
# Each feature is expanded to [x, x^2], then trained with the same RMSProp routine as minibatch().
# Returned w has shape (2 * n_features, 1): the first half are the linear terms, the second half the squared terms.
# To predict, expand the test input the same way: np.concatenate([test_x, test_x**2], axis=1).
def minibatch_2(x, y, config):
    x2 = np.concatenate([x, x ** 2], axis=1)
    return minibatch(x2, y, config)

## Training config

In [5]:
from argparse import Namespace

# RMSProp with a constant lr keeps jittering around the optimum, so a small lr + larger batch + many epochs is used to land
# very close to the closed-form least-squares solution (checked numerically).
train_config = Namespace(
    batch_size = 64,
    lr = 0.0003,
    epoch = 300,
    decay_rate = 0.9
)

## Feature normalization

In [6]:
# TODO: Normalize each column (except PM2.5) for the report (use z-score normalization)
def normalize_train_data(df):
    """
    Steps:
    1. For each column (except PM2.5): calculate mean and std
    2. Apply standardization: (column - mean) / std
    3. Store normalization parameters for later use on test data

    Returns:
        normalized_df: DataFrame with normalized features
        norm_params: Dict with {'column': {'mean': X, 'std': Y}}

    Hint: Loop through data.columns, skip PM2.5
    """
    data_norm = df.copy()
    norm_params = {}

    for col in df.columns:
        if col == 'PM2.5':  # keep PM2.5 in its raw unit (it is also the prediction target)
            continue
        mean = df[col].mean()
        std = df[col].std()
        data_norm[col] = (df[col] - mean) / std
        norm_params[col] = {'mean': mean, 'std': std}

    return data_norm, norm_params

## Feature selection

In [7]:
# Choose your features to train.
# Hint:
# 1. You can select more than one feature.
# 2. You should select "good" features.

# TODO: Carefully justify which feature should be chosen.
# Index: 0 AMB_TEMP, 1 CO, 2 NO, 3 NO2, 4 NOx, 5 O3, 6 PM10, 7 WS_HR, 8 RAINFALL, 9 RH, 10 SO2,
#        11 WD_HR, 12 WIND_DIREC, 13 WIND_SPEED, 14 PM2.5
# Selected: PM10 and PM2.5 (6, 14) only.
# Justification:
# - The test set comes from a cleaner, noisier-at-hour-level regime than train (mean PM2.5 7.8 vs 11.6, SO2 0.9 vs 1.7, RH 81 vs 73,
#   lag-1 autocorrelation of PM2.5 0.60 vs 0.85).
# - Wind-direction columns are circular (0 and 359 degrees are neighbours), so a linear model cannot use them anyway.

#feats = [14]
feats = [6, 14]

## Training

In [8]:
# Training data preprocessing
def train_processing(train_df, norm=False):
    """Process training train_df with optional normalization"""

    if norm:
        # Normalize training data and save parameters (mean & std)
        data_norm, norm_params = normalize_train_data(train_df)
        data_values = data_norm.values
    else:
        # Use raw training data
        data_values = train_df.values
        norm_params = None

    # Common processing steps
    train_data = np.transpose(np.array(np.float64(data_values)))
    train_x, train_y = parse2train(train_data, feats)

    return train_x, train_y, norm_params

train_x, train_y, norm_params = train_processing(train_df, norm=True)

# Train your regression model
w, bias = minibatch(train_x, train_y, train_config)
print(w.shape, bias.shape)

(16, 1) (1,)


## Testing

In [9]:
def parse2test(data, feats):
  x = []
  for i in range(90):
    x_tmp = data[feats,8*i: 8*i+8]
    x.append(x_tmp.reshape(-1,))

  # x.shape: (n, 15, 8)
  x = np.array(x)
  return x

def normalize_test_data(df, norm_params):
    data_norm = df.copy()

    for col, params in norm_params.items():
        if col in df.columns:
            data_norm[col] = (df[col] - params['mean']) / params['std']

    return data_norm

In [10]:
# Testing data preprocessing
def test_processing(test_df, norm=False, norm_params=norm_params):
    if norm:
        if norm_params is None:
            raise ValueError("norm_params required when norm=True")

        # Apply training normalization parameters to testing data
        data_norm = normalize_test_data(test_df, norm_params)
        data_values = data_norm.values
    else:
        # Use raw testing data
        data_values = test_df.values

    # Common processing steps
    test_data = np.transpose(np.array(np.float64(data_values)))
    test_x = parse2test(test_data, feats)

    return test_x

test_x = test_processing(test_df, norm=True, norm_params=norm_params)

## Write main result as .csv  (`my_sol.csv`)

In [11]:
with open('my_sol.csv', 'w', newline='') as csvf:
    writer = csv.writer(csvf)
    writer.writerow(['Id','Predicted'])

    print(test_x.shape)
    for i in range(int(test_x.shape[0])):
      # Prediction of linear regression
      prediction = (np.dot(np.reshape(w,-1),test_x[i]) + bias)[0]
      writer.writerow([i, prediction])

from google.colab import files
files.download('my_sol.csv')

(90, 16)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Part 2 - Experiments for the report**

Nothing below changes the Part 1 functions or globals. Each experiment reseeds (9487), trains with the stated parameters, and writes `<id>.csv` (`Id,Predicted`). The Kaggle public RMSE in the table is the score obtained when that csv was uploaded.

Common settings unless stated otherwise: seed 9487, RMSProp, window = 8 hours -> next-hour PM2.5.

| id | report | features | z-score | filter | model | config |
|---|---|---|---|---|---|---|
| q1_row1 | Q1 row 1 | CO,O3,PM10,RH,WIND_SPEED,PM2.5 | yes | target > 40 dropped | linear | A |
| q1_row2 | Q1 row 2 | CO,O3,PM10,RH,WIND_SPEED,PM2.5 | yes | target or any input PM2.5 > 30 dropped | linear | A |
| q1_row3 | Q1 row 3 | PM2.5 | yes | same as row 2 | linear | A |
| q1_row4 | Q1 row 4 | PM10,PM2.5 | yes | none | linear | A |
| q1_row5 | Q1 row 5 (= `my_sol.csv`) | PM10,PM2.5 | yes | same as row 2 | linear | A |
| q2_linear_NO | Q2(b) | NO | yes | `valid()` (target <= 30) | linear | A |
| q2_poly2_NO | Q2(b) | NO | yes | `valid()` (target <= 30) | 2nd-order | A |
| q3_cfgA/B/C_norm, _nonorm | Q3(b) | WD_HR,PM2.5 | yes / no | same as row 2 | linear | A / B / C |

Configs: **A** = `train_config` (batch 64, lr 3e-4, 300 epochs) | **B** = batch 32, lr 3e-3, 100 epochs | **C** = TA sample config, batch 3, lr 0.1, 5 epochs (decay_rate 0.9 for all).

In [12]:
# ---------------- helpers (do not touch the Part 1 globals) ----------------
import csv

Q_SEED = 9487

# Filters. valid_pm25 = the report's rule "keep a window only if the target AND all 8 input PM2.5 are <= 30", applied to the
# PM2.5 row only (always the LAST row of feats), so it also works when the other features are not z-scored.
def valid_pm25(x, y):
    return y <= 30 and x[-1].max() <= 30

def valid_none(x, y):
    return True

def valid_target40(x, y):
    return y <= 40

# Configs
cfg_A = train_config
cfg_B = Namespace(batch_size=32, lr=0.003, epoch=100, decay_rate=0.9)
cfg_C = Namespace(batch_size=3,  lr=0.1,   epoch=5,   decay_rate=0.9)

# parameterised copy of parse2train (features / filter passed in instead of read from globals)
def q_parse2train(data, feats_, valid_fn):
    x, y = [], []
    for i in range(data.shape[1] - 8):
        x_tmp = data[feats_, i:i+8]
        y_tmp = data[-1, i+8]                       # PM2.5 is the last column
        if valid_fn(x_tmp, y_tmp):
            x.append(x_tmp.reshape(-1,))
            y.append(y_tmp)
    return np.array(x), np.array(y)

def run_experiment(tag, feats_, norm, cfg, valid_fn, poly=False):
    """Preprocess -> train -> predict the 90 test rows -> write <tag>.csv"""
    if norm:
        tr_df, params = normalize_train_data(train_df)
        te_df = normalize_test_data(test_df, params)
    else:
        tr_df, te_df = train_df, test_df
    tr = np.transpose(np.array(np.float64(tr_df.values)))
    te = np.transpose(np.array(np.float64(te_df.values)))
    train_x, train_y = q_parse2train(tr, feats_, valid_fn)
    test_x = parse2test(te, feats_)

    np.random.seed(Q_SEED)                          # reseed: every experiment is reproducible on its own
    w, bias = (minibatch_2 if poly else minibatch)(train_x, train_y, cfg)
    w, b = np.reshape(w, -1), float(np.ravel(bias)[0])

    expand = (lambda a: np.concatenate([a, a ** 2], axis=1)) if poly else (lambda a: a)
    pred = expand(test_x) @ w + b
    train_rmse = float(np.sqrt(np.mean((expand(train_x) @ w + b - train_y) ** 2)))

    with open(f'{tag}.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Id', 'Predicted'])
        for i, p in enumerate(pred):
            writer.writerow([i, p])
    if not np.isfinite(pred).all():
        print(f'!! {tag}: non-finite predictions (training diverged)')
    return dict(w=w, b=b, pred=pred, n_train=len(train_y), train_rmse=train_rmse)

## Data checks quoted in Q1

In [13]:
tr_raw = np.transpose(np.array(np.float64(train_df.values)))
print('corr(PM10, PM2.5) on train      :', round(train_df['PM10'].corr(train_df['PM2.5']), 4))
print('mean PM2.5  train / test        :', round(train_df['PM2.5'].mean(), 2), '/', round(test_df['PM2.5'].mean(), 2))
print('max  PM2.5  train / test        :', train_df['PM2.5'].max(), '/', test_df['PM2.5'].max())
print('windows total / kept by filter  :', len(q_parse2train(tr_raw, [14], valid_none)[1]), '/', len(q_parse2train(tr_raw, [14], valid_pm25)[1]))

corr(PM10, PM2.5) on train      : 0.8189
mean PM2.5  train / test        : 11.58 / 7.84
max  PM2.5  train / test        : 80 / 30.0
windows total / kept by filter  : 5766 / 5170


## Experiment table (parameters + Kaggle public RMSE) and run

In [14]:
# Feature indices: 0 AMB_TEMP, 1 CO, 2 NO, 3 NO2, 4 NOx, 5 O3, 6 PM10, 7 WS_HR, 8 RAINFALL, 9 RH, 10 SO2,
#                 11 WD_HR, 12 WIND_DIREC, 13 WIND_SPEED, 14 PM2.5
SIX = [1, 5, 6, 9, 13, 14]      # CO, O3, PM10, RH, WIND_SPEED, PM2.5

EXPERIMENTS = [
    # id               report   features  norm   filter         poly   config  Kaggle public RMSE
    # ---- Q1: data preprocessing ----
    ('q1_row1',        'Q1',    SIX,      True,  valid_target40, False, 'A',   3.53749),
    ('q1_row2',        'Q1',    SIX,      True,  valid_pm25,     False, 'A',   3.37803),
    ('q1_row3',        'Q1',    [14],     True,  valid_pm25,     False, 'A',   3.13460),
    ('q1_row4',        'Q1',    [6, 14],  True,  valid_none,     False, 'A',   3.05881),
    ('q1_row5',        'Q1',    [6, 14],  True,  valid_pm25,     False, 'A',   3.05207),   # = Part 1 / my_sol.csv
    # ---- Q2: 2nd-order polynomial, NO only (filter = the notebook's valid()) ----
    ('q2_linear_NO',   'Q2',    [2],      True,  valid,          False, 'A',   4.03963),
    ('q2_poly2_NO',    'Q2',    [2],      True,  valid,          True,  'A',   4.39283),
    # ---- Q3(b): WD_HR + PM2.5, normalisation on / off ----
    ('q3_cfgA_norm',   'Q3',    [11, 14], True,  valid_pm25,     False, 'A',   3.03433),
    ('q3_cfgA_nonorm', 'Q3',    [11, 14], False, valid_pm25,     False, 'A',   3.04134),
    ('q3_cfgB_norm',   'Q3',    [11, 14], True,  valid_pm25,     False, 'B',   3.03409),
    ('q3_cfgB_nonorm', 'Q3',    [11, 14], False, valid_pm25,     False, 'B',   3.03863),
    ('q3_cfgC_norm',   'Q3',    [11, 14], True,  valid_pm25,     False, 'C',   6.04676),
    ('q3_cfgC_nonorm', 'Q3',    [11, 14], False, valid_pm25,     False, 'C',   31.26913),
]
CFGS = {'A': cfg_A, 'B': cfg_B, 'C': cfg_C}

results, rows = {}, []
for tag, q, feats_, norm, vfn, poly, cname, kaggle in EXPERIMENTS:
    r = run_experiment(tag, feats_, norm, CFGS[cname], vfn, poly)
    results[tag] = r
    cfg = CFGS[cname]
    rows.append(dict(id=tag, report=q, features=','.join(train_df.columns[j] for j in feats_), zscore=norm,
                     filter=vfn.__name__, model='2nd-order' if poly else 'linear', config=cname,
                     batch=cfg.batch_size, lr=cfg.lr, epochs=cfg.epoch, n_train=r['n_train'],
                     train_rmse=round(r['train_rmse'], 3), kaggle_public=kaggle))
summary = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
display(summary)

# Sanity check: q1_row5 must equal the Part 1 output
print('max |q1_row5 - my_sol.csv| =', np.abs(pd.read_csv('my_sol.csv')['Predicted'].values - results['q1_row5']['pred']).max())

,id,report,features,zscore,filter,model,config,batch,lr,epochs,n_train,train_rmse,kaggle_public
0,q1_row1,Q1,"CO,O3,PM10,RH,WIND_SPEED,PM2.5",True,valid_target40,linear,A,64,0.0003,300,5678,3.226,3.53749
1,q1_row2,Q1,"CO,O3,PM10,RH,WIND_SPEED,PM2.5",True,valid_pm25,linear,A,64,0.0003,300,5170,2.938,3.37803
2,q1_row3,Q1,PM2.5,True,valid_pm25,linear,A,64,0.0003,300,5170,3.114,3.13460
3,q1_row4,Q1,"PM10,PM2.5",True,valid_none,linear,A,64,0.0003,300,5766,3.621,3.05881
4,q1_row5,Q1,"PM10,PM2.5",True,valid_pm25,linear,A,64,0.0003,300,5170,3.038,3.05207
5,q2_linear_NO,Q2,NO,True,valid,linear,A,64,0.0003,300,5503,6.893,4.03963
6,q2_poly2_NO,Q2,NO,True,valid,2nd-order,A,64,0.0003,300,5503,6.989,4.39283
7,q3_cfgA_norm,Q3,"WD_HR,PM2.5",True,valid_pm25,linear,A,64,0.0003,300,5170,3.089,3.03433
8,q3_cfgA_nonorm,Q3,"WD_HR,PM2.5",False,valid_pm25,linear,A,64,0.0003,300,5170,3.096,3.04134
9,q3_cfgB_norm,Q3,"WD_HR,PM2.5",True,valid_pm25,linear,B,32,0.0030,100,5170,3.091,3.03409


max |q1_row5 - my_sol.csv| = 5.329070518200751e-15


## Q2(b) - learned parameters (NO only, 2nd-order)

In [15]:
q2 = results['q2_poly2_NO']
display(pd.DataFrame({'hour (0 = oldest)': range(8), 'w1 (x)': q2['w'][:8], 'w2 (x^2)': q2['w'][8:]}).round(4))
print('bias b =', round(q2['b'], 4))
print('Kaggle public: 1st order 4.03963 | 2nd order 4.39283')

,hour (0 = oldest),w1 (x),w2 (x^2)
0,0,-0.0639,0.0422
1,1,-0.0466,0.0682
2,2,0.0430,0.0829
3,3,0.2173,-0.0036
4,4,0.1358,0.0273
5,5,-0.3408,0.1042
6,6,-0.1883,0.0643
7,7,-0.6058,0.1587


bias b = 7.5856
Kaggle public: 1st order 4.03963 | 2nd order 4.39283


## **Final submissions - the two best Kaggle results**

Ranked by Kaggle public RMSE:

| rank | file | setting | Kaggle public |
|---|---|---|---|
| **1** | **`q3_cfgB_norm.csv`** | WD_HR + PM2.5, z-score, config B (batch 32, lr 0.003, 100 epochs) | **3.03409** |
| **2** | **`q3_cfgA_norm.csv`** | WD_HR + PM2.5, z-score, config A = `train_config` (batch 64, lr 0.0003, 300 epochs) | **3.03433** |

(The Part 1 model, `my_sol.csv` = `q1_row5.csv`, PM10 + PM2.5, scores 3.05207.)

In [16]:
best = summary.sort_values('kaggle_public').head(2)
display(best[['id', 'features', 'zscore', 'config', 'batch', 'lr', 'epochs', 'kaggle_public']])

from google.colab import files
for t in best['id']:
    files.download(f'{t}.csv')

,id,features,zscore,config,batch,lr,epochs,kaggle_public
9,q3_cfgB_norm,"WD_HR,PM2.5",True,B,32,0.0030,100,3.03409
7,q3_cfgA_norm,"WD_HR,PM2.5",True,A,64,0.0003,300,3.03433


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>